# 04 — Structured outputs and tool calling

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mano-octavianojr/chatbot-for-insurance-knowledge/blob/main/notebooks/04_structured_outputs_and_tools.ipynb)

Run cells from top to bottom. This lesson uses fictional insurance examples for education, not coverage advice. Do not enter personal, medical, or real policy information. No GPU is needed.

## Goal
Explore two techniques beyond basic chat: structured JSON answers and calling a Python function. Both use the Gemini API. Structured output helps applications process answers; tool calling delegates calculations to code. These techniques can also be combined with RAG.

Local models and fine-tuning are discussed at the end as separate extensions, because they introduce hardware and training requirements.

## Connect to Gemini

Create a key in [Google AI Studio](https://aistudio.google.com/apikey). In Colab, open **Secrets** (key icon), add `GEMINI_API_KEY`, and enable notebook access. Never paste the key into a code cell. Outside Colab, use an environment variable with that name.

Gemini offers a free tier for eligible models, accounts, and regions; quotas, availability, and pricing can change. Check [pricing](https://ai.google.dev/gemini-api/docs/pricing) and [rate limits](https://ai.google.dev/gemini-api/docs/rate-limits) before running. Your prompts and documents are sent to Google; review its data-use terms. These cells need Internet access.

In [ ]:
%pip -q install "google-genai>=2.0,<3.0"

In [ ]:
import os
from google import genai
from google.genai import types

def read_api_key():
    key = os.environ.get("GEMINI_API_KEY")
    if key:
        return key
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
    except ImportError:
        raise RuntimeError("Set GEMINI_API_KEY in your environment.") from None
    except Exception:
        raise RuntimeError("Add GEMINI_API_KEY to Colab Secrets and enable notebook access.") from None
    if not key:
        raise RuntimeError("GEMINI_API_KEY is missing.")
    return key

client = genai.Client(api_key=read_api_key())
# Change this if your account requires a different supported model.
MODEL = "gemini-flash-latest"


If a request fails: check key permissions for authentication errors, choose an available model for model errors, and wait or reduce requests for quota errors (429). Do not repeatedly retry unchanged requests. Successful responses do not guarantee factual accuracy.

## Structured output
Request a small JSON schema and validate it locally. Schema compliance does not guarantee factual correctness.

In [ ]:
import json
schema = {
    "type": "object",
    "properties": {
        "answer": {"type": "string"},
        "needs_clarification": {"type": "boolean"},
        "follow_up_question": {"type": "string"},
    },
    "required": ["answer", "needs_clarification", "follow_up_question"],
}
response = client.models.generate_content(
    model=MODEL,
    contents="Is my procedure covered?",
    config=types.GenerateContentConfig(
        system_instruction="Explain insurance generally. For ambiguous coverage questions, ask for non-sensitive clarification. Do not determine real coverage.",
        response_mime_type="application/json",
        response_json_schema=schema,
    ),
)
result = json.loads(response.text)
assert isinstance(result["answer"], str)
assert isinstance(result["needs_clarification"], bool)
assert isinstance(result["follow_up_question"], str)
print(json.dumps(result, indent=2))


## Define a deterministic tool
This fictional calculator handles dollars in integer cents to avoid floating-point money errors. It assumes one covered claim and no coinsurance. Real contracts can differ. Local assertions verify the calculation without an API key.

In [ ]:
def calculate_example_payment(covered_cost_cents: int, remaining_deductible_cents: int, benefit_limit_cents: int) -> dict:
    """Calculate fictional claim amounts in integer cents, without coinsurance.

    Args:
        covered_cost_cents: Covered claim cost in cents.
        remaining_deductible_cents: Remaining deductible in cents.
        benefit_limit_cents: Remaining insurer benefit limit in cents.
    """
    values = [covered_cost_cents, remaining_deductible_cents, benefit_limit_cents]
    if any(type(value) is not int or value < 0 for value in values):
        raise ValueError("Inputs must be nonnegative integer cents.")
    deductible_paid = min(covered_cost_cents, remaining_deductible_cents)
    insurer_paid = min(covered_cost_cents - deductible_paid, benefit_limit_cents)
    return {"insurer_paid_cents": insurer_paid,
            "customer_paid_cents": covered_cost_cents - insurer_paid}

assert calculate_example_payment(120000, 50000, 1000000) == {
    "insurer_paid_cents": 70000, "customer_paid_cents": 50000}
assert calculate_example_payment(20000, 50000, 1000000)["insurer_paid_cents"] == 0
assert calculate_example_payment(120000, 50000, 30000)["customer_paid_cents"] == 90000
print("Three calculator checks passed.")


## Let Gemini select the tool
The SDK can execute supplied Python functions automatically and send results back to the model. Expose only functions you intend to allow. This tool performs a calculation and has no external side effects. Inspect the recorded call as well as the answer.

In [ ]:
response = client.models.generate_content(
    model=MODEL,
    contents="Use the calculator: covered cost is $1,200, remaining deductible is $500, and remaining benefit limit is $10,000. Explain both payment amounts in dollars.",
    config=types.GenerateContentConfig(
        system_instruction="Use calculate_example_payment for payment calculations. Convert dollars to integer cents for the tool and back to dollars for the answer. Explain that the calculation is fictional.",
        tools=[calculate_example_payment],
        automatic_function_calling=types.AutomaticFunctionCallingConfig(maximum_remote_calls=2),
    ),
)
print(response.text or "No text returned.")
# Show actual tool requests, so an answer alone is not mistaken for a tool check.
called = []
for turn in response.automatic_function_calling_history or []:
    for part in turn.parts or []:
        if part.function_call:
            called.append(part.function_call.name)
            print("Tool:", part.function_call.name, "Arguments:", part.function_call.args)
assert "calculate_example_payment" in called, "The model did not call the calculator; inspect the response."


## Exercises and further techniques
- Change the cost and verify the result against a direct Python call.
- Combine lesson 3’s retrieved sources with a structured answer containing source IDs.
- Add keyword search to RAG and compare source recall with embedding-only search.

### Local open-source models
A separate extension can use a small instruction model through Transformers. It needs model downloads, license review, and sufficient RAM or a Colab GPU. Colab GPU availability is not guaranteed. Local inference removes the API requirement but not compute or model-quality constraints.

### Fine-tuning
Training on examples can improve a particular format or behavior. It requires a dataset, evaluation, and a supported training platform. Use retrieval for frequently changing policy facts; do not treat fine-tuning as a replacement for document grounding.

**Conclusion:** rules provide predictable responses; API chat adds flexible language; RAG adds document context; schemas and tools add application structure and reliable calculations.